# geometry4-shelf vs geometry6-shelf: does an FNO inherit the field representation's difficulty?**Sec 9.15c test.** Trains an FNO on both shelf-layout geometries under 5-fold CV and askswhich representation its accuracy tracks.This notebook runs the one experiment `docs/report.md` Sec 9.15c makes a falsifiableprediction about. It is **not** an architecture search and not an attempt to win anything.## The prediction being testedSec 9.15c found that "linearly solvable" is a property of the **input representation**, not ofthe dataset. On the *same* 45 geometry4-shelf files:| representation | who gets it | linear spatial R2 ||---|---|---|| compact vector (block powers + positions + HTC + ambient, ~10 scalars) | `scripts/baselines.py` ridge | **-0.667** || full per-cell power field | the linearity audit, and an FNO | **0.962** |geometry6-shelf is hard in **both** representations (0.536 from the field).An FNO consumes the *field*. So:> **Prediction.** FNO accuracy should track the difficulty of the task *in the field> representation* (0.94 on geometry4 vs 0.51 on geometry6 -- a gap of ~0.43), **not** the> difficulty in the compact representation ridge sees (-0.67 vs -2.87 -- a gap of ~2.2).**Why the test is framed on absolute R2 and not on "margin over ridge".** A first draft ofthis notebook tested whether the FNO's *margin over ridge* is larger on geometry4. That testis broken: ridge scores -2.87 on geometry6 against -0.67 on geometry4, so **any** competentmodel beats it by more on geometry6 mechanically, whatever the representation story is. Themargin is confounded by how bad the baseline is. What Sec 9.15c actually predicts is that theFNO inherits the *field* representation's difficulty ordering, so that is what is tested:| quantity | geometry4 | geometry6 | gap ||---|---|---|---|| ridge, compact representation (5-fold CV) | -0.674 | -2.866 | 2.19 || linear, field representation (5-fold CV) | **+0.941** | **+0.513** | **0.43** || FNO, field representation | ? | ? | ? |If the FNO's gap is near **0.43**, it tracks the field representation -- Sec 9.15c holds. If itis near **2.19**, it tracks the compact representation, which would be strange and would meansomething other than representation governs difficulty. If the FNO lands well *below* thelinear-on-field ceiling on both, the field representation is not usable at n=45 and theinteresting comparison is FNO vs linear-on-field, not FNO vs ridge.**This can fail in three informative ways:**1. FNO gap near 2.19 rather than 0.43 -> the representation story is wrong; difficulty is not   representation-mediated.2. FNO loses to ridge on both -> consistent with Sec 9.12d, where ridge beat both FNO configs   on every hotspot metric under CV; would say the field representation is unusable at n=45.3. FNO wins on both by similar margins -> representation matters but does not explain the   geometry4/geometry6 difference.Record whichever happens. A null result is a result, and Sec 9.15c should be narrowed if theprediction fails.## Protocol (fixed so results are comparable to Sec 9.15b)- **5-fold CV over all 45 scenarios**, folds from `kfold_indices(n, 5, seed=0)` copied  byte-for-byte from `scripts/hotspot_eval.py`, so the FNO is scored on the *same held-out  fields* as the ridge numbers already in the report.- **Detrended metrics** copied byte-for-byte from `scripts/layout_cv.py::per_scenario_stats`.  Raw MAE is dominated by the ~325 K offset (Sec 9.1) and must not be used.- **Identical epoch budget for both geometries.** Unequal budgets caused the convergence  confound in Sec 9.12.- Ridge is recomputed **in this notebook on the same folds**, not quoted, so the comparison  cannot drift from a stale number.## Data you must attach**`src/` is cloned from GitHub** -- the repo is public and tracks it, so no upload is needed.Set **Settings -> Internet -> On**. (If internet is off, the notebook falls back to anattached dataset containing `src/`.)**The data must still be attached.** `data/` is gitignored, so the `.npz` files are *not* inthe repo:| dataset | contents | size ||---|---|---|| geometry4 shelf | 45 `.npz` from `data/3d-ice-layout-geometry4/geometry4/` | ~12 MB || geometry6 shelf | 45 `.npz` from `data/3d-ice-layout-geometry6/geometry6/` | ~31 MB |Slugs do not matter -- both are located by searching attached datasets for their contents.One combined dataset works as well as two.Enable **GPU T4 x1**. Expect ~2-4 h; geometry6 is 2.5x the grid of geometry4(56x168x15 = 141,120 cells vs 100x56x10 = 56,000).

In [ ]:
import subprocess, syssubprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)import torchprint('Python :', sys.version.split()[0])print('PyTorch:', torch.__version__)print('CUDA   :', torch.cuda.is_available())if torch.cuda.is_available():    print('GPU    :', torch.cuda.get_device_name(0))    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')else:    print('WARNING: no GPU. Enable Settings -> Accelerator -> GPU T4 x1 before training cells.')

In [ ]:
import sys, subprocessfrom pathlib import Path# ---------------------------------------------------------------------------# src/ comes from GitHub; the .npz data does not.##   github.com/rajul-kk/thermo-3dic-surrogates is public and TRACKS src/, so the#   code is cloned rather than uploaded -- no Kaggle dataset needed for it, and#   it is never a stale zip. Requires Settings -> Internet: On. If internet is#   off we fall back to finding src/ in an attached dataset.##   data/ is gitignored (see .gitignore), so the 45+45 .npz files are NOT in the#   repo and must still be attached as Kaggle dataset(s). They are ~43 MB total.#   Slugs do not matter: they are located by content.# ---------------------------------------------------------------------------REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'INPUT = Path('/kaggle/input')OUT_DIR = Path('/kaggle/working/checkpoints/layout_repr_test')OUT_DIR.mkdir(parents=True, exist_ok=True)print('Attached datasets under /kaggle/input:')if INPUT.exists() and any(INPUT.iterdir()):    for d in sorted(INPUT.iterdir()):        n_npz = len(list(d.rglob('*.npz')))        has_src = bool(list(d.rglob('src/fno/model.py')))        tag = []        if has_src:            tag.append('HAS src/')        if n_npz:            tag.append(str(n_npz) + ' .npz')        print('  {:<42} {}'.format(d.name, ' | '.join(tag) if tag else '(neither src/ nor .npz)'))else:    print('  (nothing attached)')# --- src/: clone from GitHub, else fall back to an attached dataset ----------SRC_ROOT = Noneclone_dir = Path('/kaggle/working/repo')if not clone_dir.exists():    r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(clone_dir)],                       capture_output=True, text=True)    if r.returncode != 0:        print('\ngit clone failed (this is expected if Settings -> Internet is Off):')        print('  ' + (r.stderr.strip().splitlines() or ['?'])[-1])if (clone_dir / 'src' / 'fno' / 'model.py').exists():    SRC_ROOT = clone_dir    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],                         capture_output=True, text=True).stdout.strip()    print('\nsrc/ cloned from GitHub at commit ' + sha)else:    for hit in INPUT.rglob('src/fno/model.py'):        SRC_ROOT = hit.parent.parent.parent        print('\nsrc/ taken from attached dataset: ' + SRC_ROOT.name)        breakassert SRC_ROOT is not None, (    '\n\nCould not obtain src/.\n'    'Easiest fix: Settings -> Internet -> On, then re-run (the repo is public and is\n'    'cloned automatically).\n'    'Offline alternative: zip the project src/ folder so the zip root CONTAINS src/\n'    '(check src/fno/model.py is inside), upload as a Kaggle dataset and attach it.')sys.path.insert(0, str(SRC_ROOT))# --- data: must be attached; located by content, not by slug -----------------def find_geom_root(geom):    best, best_n = None, 0    for d in (INPUT.iterdir() if INPUT.exists() else []):        n = len(list(d.rglob(geom + '_*.npz')))        if n > best_n:            best, best_n = d, n    return best, best_nFILES = {}for geom in ['geometry4', 'geometry6']:    root, n = find_geom_root(geom)    assert root is not None and n > 0, (        '\n\nNo attached dataset contains ' + geom + '_*.npz.\n'        'data/ is gitignored, so these are NOT in the GitHub repo and must be uploaded.\n'        'Upload the 45 .npz files from data/3d-ice-layout-' + geom + '/' + geom + '/ as a\n'        'Kaggle dataset and attach it (~12 MB for geometry4, ~31 MB for geometry6).\n'        'The slug does not matter. See the list printed above for what is attached.'    )    FILES[geom] = sorted(root.rglob(geom + '_*.npz'))    flag = '' if len(FILES[geom]) == 45 else '   <-- expected 45!'    print('{}: {} scenarios from {}{}'.format(geom, len(FILES[geom]), root.name, flag))

In [ ]:
import logging, time, jsonimport numpy as npimport torchimport matplotlib.pyplot as pltfrom src.core.geometry_builders import get_geometry_by_namefrom src.pinn.data_loader import compute_norm_statsfrom src.fno.model import build_fnofrom src.fno.data_loader import FNODataset, predict_to_flatfrom src.fno.trainer import FNOTrainerlogging.basicConfig(level=logging.INFO,                    format='%(asctime)s %(levelname)-7s %(name)s: %(message)s',                    datefmt='%H:%M:%S')print('Imports OK')

In [ ]:
# ---------------------------------------------------------------------------# Folds and metrics, copied byte-for-byte from the repo so the numbers printed# here are directly comparable to docs/report.md Sec 9.15b. Do not "improve"# these -- if they differ from the repo, the comparison is void.# ---------------------------------------------------------------------------def kfold_indices(n, folds, seed):    '''scripts/hotspot_eval.py::kfold_indices'''    idx = np.arange(n)    np.random.default_rng(seed).shuffle(idx)    return [np.asarray(part) for part in np.array_split(idx, folds)]def per_scenario_stats(y, p):    '''scripts/layout_cv.py::per_scenario_stats -- detrended, spatial structure only.'''    dy, dp = y - y.mean(), p - p.mean()    ss = float(np.sum(dy ** 2))    return {'det_mae': float(np.mean(np.abs(dy - dp))),            'sigma': float(dy.std()),            'r2': float(1.0 - np.sum((dy - dp) ** 2) / ss) if ss > 0 else float('nan'),            'corr': float(np.corrcoef(dp, dy)[0, 1]) if dp.std() > 0 and dy.std() > 0 else 0.0}def summarise(rows, label):    r2 = np.array([r['r2'] for r in rows])    dm = float(np.mean([r['det_mae'] for r in rows]))    sig = float(np.mean([r['sigma'] for r in rows]))    return {'model': label, 'n': len(rows), 'det_mae': dm, 'sigma': sig,            'norm_err': dm / sig, 'r2_mean': float(r2.mean()),            'r2_median': float(np.median(r2)), 'n_negative_r2': int((r2 < 0).sum()),            'corr': float(np.mean([r['corr'] for r in rows]))}FOLDS, SEED = 5, 0print(f'{FOLDS}-fold CV, seed {SEED} -- identical folds to the report')

In [ ]:
# ---------------------------------------------------------------------------# Ridge on the COMPACT representation, recomputed rather than quoted. Mirrors# scripts/baselines.py: per-block mean powers + per-block positions + HTC +# 1/HTC + ambient + TSV density. Sec 9.15c's point is that this representation# is impoverished once the layout varies, so it must be built as the report did.# ---------------------------------------------------------------------------def load_scenario(path):    d = np.load(path, allow_pickle=True)    return {'name': path.stem, 'temp': d['temp'].astype(np.float64),            'power': d['power'].astype(np.float64),            'meta': dict(d['metadata'][0])}def collect_keys(scen, prefix):    keys = set()    for s in scen:        keys |= {k for k in s['meta'] if k.startswith(prefix)}    return sorted(keys)def feature_vector(meta, block_keys, pos_keys):    htc = float(meta.get('htc', 0.0))    f = [float(meta.get(k, 0.0)) for k in block_keys]    f += [float(meta.get(k, 0.0)) for k in pos_keys]    f += [htc, 1.0 / htc if htc > 0 else 0.0,          float(meta.get('t_ambient_kelvin', 298.15)),          float(meta.get('tsv_density', 0.0))]    return np.asarray(f, dtype=np.float64)def ridge_cv(scen, lam=1.0):    bk = collect_keys(scen, 'block_power_')    if not bk:        bk = [k for k in sorted(scen[0]['meta']) if k.startswith('block_') and 'power' in k]    pk = collect_keys(scen, 'block_x_') + collect_keys(scen, 'block_y_')    X = np.stack([feature_vector(s['meta'], bk, pk) for s in scen])    Y = np.stack([s['temp'] for s in scen])    rows = []    for fold in kfold_indices(len(scen), FOLDS, SEED):        te = set(fold.tolist())        tr = np.array([i for i in range(len(scen)) if i not in te])        Xtr, Ytr = X[tr], Y[tr]        keep = Xtr.std(0) > 1e-12                      # drop constant columns        mu, sd = Xtr[:, keep].mean(0), Xtr[:, keep].std(0) + 1e-12        A = np.hstack([(Xtr[:, keep] - mu) / sd, np.ones((len(tr), 1))])        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Ytr)        B = np.hstack([(X[fold][:, keep] - mu) / sd, np.ones((len(fold), 1))])        P = B @ W        for j, i in enumerate(fold):            rows.append(per_scenario_stats(Y[i], P[j]))    return rows, int(keep.sum())SCEN = {g: [load_scenario(p) for p in FILES[g]] for g in FILES}RIDGE = {}for g in SCEN:    rows, nfeat = ridge_cv(SCEN[g])    RIDGE[g] = summarise(rows, 'ridge (compact)')    print(f'{g}: ridge over {nfeat} non-constant compact features -> '          f"R2 mean {RIDGE[g]['r2_mean']:+.3f}  median {RIDGE[g]['r2_median']:+.3f}  "          f"norm_err {RIDGE[g]['norm_err']:.3f}  R2<0 {RIDGE[g]['n_negative_r2']}/45")print()print('SANITY CHECK vs docs/report.md Sec 9.15b: geometry4 R2 mean should be near -0.667,')print('geometry6 near -2.866 (median near -0.303). A large deviation means the folds or')print('features differ from the report -- fix that before reading anything else here.')

In [ ]:
# ---------------------------------------------------------------------------# Linear fit on the FIELD representation -- the same input the FNO gets. This is# the Sec 9.15c row reading 0.962 (geometry4) and 0.536 (geometry6). Included so# the FNO is compared against the best LINEAR use of its own input; otherwise a# win is ambiguous between "the network helped" and "the representation helped".# ---------------------------------------------------------------------------def linear_field_cv(scen, pca_k=8, lam=1e-2):    X = np.stack([s['power'] for s in scen])    Y = np.stack([s['temp'] for s in scen])    rows = []    for fold in kfold_indices(len(scen), FOLDS, SEED):        te = set(fold.tolist())        tr = np.array([i for i in range(len(scen)) if i not in te])        Xtr = X[tr]        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12        Xc = (Xtr - mu) / sd        # economy SVD of the (n_train x n_cells) matrix; n_train is 36, so cheap        _, _, Vt = np.linalg.svd(Xc, full_matrices=False)        k = min(pca_k, Vt.shape[0])        B = Vt[:k].T        A = np.hstack([Xc @ B, np.ones((len(tr), 1))])        W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Y[tr])        F = np.hstack([((X[fold] - mu) / sd) @ B, np.ones((len(fold), 1))])        P = F @ W        for j, i in enumerate(fold):            rows.append(per_scenario_stats(Y[i], P[j]))    return rowsLINFIELD = {}for g in SCEN:    LINFIELD[g] = summarise(linear_field_cv(SCEN[g]), 'linear (field)')    print(f"{g}: linear-on-field -> R2 mean {LINFIELD[g]['r2_mean']:+.3f}  "          f"median {LINFIELD[g]['r2_median']:+.3f}  norm_err {LINFIELD[g]['norm_err']:.3f}")

In [ ]:
# ---------------------------------------------------------------------------# FNO configuration. IDENTICAL for both geometries except grid shape, which is a# property of the data. Equal epochs is the control Sec 9.12 found missing.# ---------------------------------------------------------------------------CHANNELS   = 32N_BLOCKS   = 4MODES      = (16, 16, 8)EPOCHS     = 300        # same for both geometriesBATCH_SIZE = 2          # geometry6 is 141k cells/scenario; keep VRAM headroomLR         = 1e-3DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')GRID = {}for g in FILES:    c = np.load(FILES[g][0], allow_pickle=True)['coords']    GRID[g] = tuple(len(np.unique(c[:, i])) for i in range(3))    print(f'{g}: grid {GRID[g]} = {int(np.prod(GRID[g])):,} cells')print()print(f'Device {DEVICE} | channels {CHANNELS} | blocks {N_BLOCKS} | modes {MODES} | epochs {EPOCHS}')

In [ ]:
def fno_cv(geom, epochs=EPOCHS):    '''5-fold CV on the SAME folds, training a fresh FNO per fold.'''    files = FILES[geom]    geoms = {geom: get_geometry_by_name(geom)}    grid = GRID[geom]    rows, times = [], []    for fi, fold in enumerate(kfold_indices(len(files), FOLDS, SEED), 1):        te = set(fold.tolist())        tr_files = [files[i] for i in range(len(files)) if i not in te]        te_files = [files[i] for i in fold]        # Normalisation from TRAIN ONLY -- using all files would leak test statistics.        norm = compute_norm_stats(tr_files, geoms)        tr_ds = FNODataset(tr_files, norm, grid)        te_ds = FNODataset(te_files, norm, grid)        model = build_fno(grid, modes=MODES, hidden_ch=CHANNELS,                          n_blocks=N_BLOCKS, device=DEVICE)        trainer = FNOTrainer(model, norm, tr_ds, te_ds,                             output_dir=OUT_DIR / f'{geom}_fold{fi}',                             batch_size=BATCH_SIZE, epochs=epochs, lr=LR, device=DEVICE)        t0 = time.time()        trainer.train()        times.append(time.time() - t0)        for j in range(len(te_ds)):            pred, true = predict_to_flat(model, te_ds.items[j], DEVICE, norm)            rows.append(per_scenario_stats(true, pred))        print(f'  {geom} fold {fi}/{FOLDS} done in {times[-1]/60:.1f} min '              f'({len(te_ds)} held-out fields)')        del model, trainer        torch.cuda.empty_cache()    s = summarise(rows, 'FNO (field)')    s['train_seconds_total'] = float(np.sum(times))    s['train_seconds_per_fold'] = float(np.mean(times))    return sFNO = {}for g in ['geometry4', 'geometry6']:          # geometry4 first: cheaper, fails faster    print(f'=== FNO on {g}-shelf ===')    FNO[g] = fno_cv(g)    print(f"{g}: FNO -> R2 mean {FNO[g]['r2_mean']:+.3f}  median {FNO[g]['r2_median']:+.3f}  "          f"norm_err {FNO[g]['norm_err']:.3f}  R2<0 {FNO[g]['n_negative_r2']}/45  "          f"({FNO[g]['train_seconds_total']/3600:.2f} GPU-h total)")    print()

In [ ]:
# ---------------------------------------------------------------------------# Verdict. Sec 9.15c predicts the FNO inherits the FIELD representation's# difficulty ordering. It is tested on the geometry4-minus-geometry6 GAP in# absolute detrended R2, NOT on the margin over ridge -- ridge is so much worse# on geometry6 (-2.87 vs -0.67) that any decent model beats it by more there# mechanically, which would confound a margin-based test.# ---------------------------------------------------------------------------import pandas as pdrecs = []for g in ['geometry4', 'geometry6']:    for s in (RIDGE[g], LINFIELD[g], FNO[g]):        recs.append({'geometry': g, 'model': s['model'], 'det_mae': s['det_mae'],                     'norm_err': s['norm_err'], 'r2_mean': s['r2_mean'],                     'r2_median': s['r2_median'], 'n_negative_r2': s['n_negative_r2'],                     'corr': s['corr']})print(pd.DataFrame(recs).to_string(index=False, float_format=lambda v: f'{v:.4f}'))gap = {k: d['geometry4']['r2_mean'] - d['geometry6']['r2_mean']       for k, d in [('ridge_compact', RIDGE), ('linear_field', LINFIELD), ('fno', FNO)]}print()print('=' * 78)print('geometry4 minus geometry6, detrended spatial R2 (mean) -- the quantity predicted')print('=' * 78)print(f"  ridge, compact representation : {gap['ridge_compact']:+.3f}")print(f"  linear, field representation  : {gap['linear_field']:+.3f}   <- Sec 9.15c predicts FNO near this")print(f"  FNO,   field representation   : {gap['fno']:+.3f}")d_field = abs(gap['fno'] - gap['linear_field'])d_compact = abs(gap['fno'] - gap['ridge_compact'])tracks = 'FIELD' if d_field < d_compact else 'COMPACT'print()print(f'FNO gap is closer to the {tracks} representation '      f'(|d_field| {d_field:.3f} vs |d_compact| {d_compact:.3f}).')print(f'PREDICTION (Sec 9.15c): {"HELD" if tracks == "FIELD" else "FAILED"}')# Secondary: does the FNO reach the linear ceiling in its own representation?print()print('Secondary -- FNO vs the best LINEAR use of the same input (field):')for g in ['geometry4', 'geometry6']:    m = FNO[g]['r2_mean'] - LINFIELD[g]['r2_mean']    verdict_g = 'above' if m > 0 else 'below'    print(f'  {g}: {m:+.3f} ({verdict_g} the linear-on-field ceiling)')print()print('If the FNO is below the linear-on-field ceiling on both, the network adds nothing over')print('a closed-form fit on its own input, and that is the headline regardless of the gap test.')out = {'folds': FOLDS, 'seed': SEED, 'epochs': EPOCHS,       'config': {'channels': CHANNELS, 'blocks': N_BLOCKS, 'modes': list(MODES),                  'batch_size': BATCH_SIZE, 'lr': LR},       'grid': {g: list(GRID[g]) for g in GRID},       'ridge_compact': RIDGE, 'linear_field': LINFIELD, 'fno': FNO,       'geometry4_minus_geometry6_gap': gap,       'fno_tracks': tracks, 'prediction_held': tracks == 'FIELD'}(OUT_DIR / 'verdict.json').write_text(json.dumps(out, indent=2))print()print(f'Saved {OUT_DIR / "verdict.json"}')

In [ ]:
# R2 distributions. The mean is heavy-tailed on these datasets (Sec 9.15b), so# the median matters as much as the centre -- both are shown.fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)for ax, g in zip(axes, ['geometry4', 'geometry6']):    ax.bar([0, 1, 2],           [RIDGE[g]['r2_mean'], LINFIELD[g]['r2_mean'], FNO[g]['r2_mean']],           color=['#bb2233', '#ee9933', '#2277aa'])    ax.errorbar([0, 1, 2],                [RIDGE[g]['r2_median'], LINFIELD[g]['r2_median'], FNO[g]['r2_median']],                fmt='k_', markersize=22, linestyle='none', label='median')    ax.set_xticks([0, 1, 2])    ax.set_xticklabels(['ridge\n(compact)', 'linear\n(field)', 'FNO\n(field)'])    ax.axhline(0, color='k', lw=0.8)    ax.set_title(f'{g}-shelf  (bar = mean, tick = median)')    ax.set_ylabel('detrended spatial R$^2$')    ax.legend()plt.tight_layout()plt.savefig(OUT_DIR / 'r2_comparison.png', dpi=130)plt.show()print('Training cost (T4):')for g in FNO:    print(f"  {g}: {FNO[g]['train_seconds_total']/3600:.2f} GPU-h total, "          f"{FNO[g]['train_seconds_per_fold']/60:.1f} min/fold x {FOLDS} folds")

## Reading the result honestly**Report whatever comes out, including a null or negative result.** Three things to resist:1. **Do not tune until the prediction passes.** If it fails at 300 epochs, that is the result   at 300 epochs. Re-running with a bigger model until Sec 9.15c is confirmed is precisely the   researcher-degrees-of-freedom failure this project documents (`docs/report.md` Sec 9.1,   Sec 9.12; McGreivy & Hakim 2024 on outcome-reporting bias).2. **Check the ridge sanity line first.** If in-notebook ridge does not land near Sec 9.15b's   -0.667 / -2.866, the folds or features differ and *no* comparison here is valid.3. **Do not read "margin over ridge" as the result.** Ridge scores -2.87 on geometry6 and   -0.67 on geometry4, so a larger margin on geometry6 is mechanical and says nothing about   representation. The gap test above exists because of that; an earlier draft of this   notebook got it wrong.**If the prediction holds**, keep the claim narrow: *on these datasets, at this budget, thebenefit of a neural operator tracks how impoverished the baseline's representation is ratherthan how hard the task is.* That is a statement about benchmark design, not about FNO quality.**Limits to state alongside any result:**- n=45 per geometry. CV uses every scenario as held-out once, which removes the single-split  lottery of Sec 9.15 but not the sparsity.- One architecture, one budget, **one seed per fold**. Sec 9.12d already showed a 5-scenario  split inverting an FNO-vs-ridge ranking, so this is the weakest part of the design: if the  two margins come out close, run 3 seeds per fold before claiming anything.- geometry4 and geometry6 differ in grid size (56k vs 141k cells) as well as layout  difficulty, so they are not perfectly matched. A margin difference could partly reflect  that, and the write-up must say so.